# City Nature Challenge - Data Collection

This notebook fetches CNC competition data for **San Diego**, **San Antonio**, and **Los Angeles**
from the [iNaturalist API](https://api.inaturalist.org/v1/docs/) and saves it to CSV files
that the backend loads at runtime.

Run this notebook once (or whenever you want to refresh the data) to regenerate the files in `backend/data/`.

In [ ]:
import json
import time
import urllib.request
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

API_BASE = "https://api.inaturalist.org/v1"

CITIES = {
    "San Diego": {
        2023: "city-nature-challenge-2023-san-diego-county",
        2024: "city-nature-challenge-2024-san-diego-county",
        2025: "city-nature-challenge-2025-san-diego-county",
    },
    "San Antonio": {
        2023: "city-nature-challenge-2023-san-antonio-metro-area",
        2024: "city-nature-challenge-2024-san-antonio-metro-area",
        2025: "city-nature-challenge-2025-san-antonio-metro-area",
    },
    "Los Angeles": {
        2023: "city-nature-challenge-2023-los-angeles-county",
        2024: "city-nature-challenge-2024-los-angeles-county",
        2025: "city-nature-challenge-2025-los-angeles-county",
    },
}

def api_get(url):
    """Fetch JSON from the iNaturalist API with a polite delay."""
    req = urllib.request.Request(url, headers={"User-Agent": "CNC-DS3-Consulting/1.0"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        data = json.loads(resp.read().decode())
    time.sleep(1)  # rate-limit courtesy
    return data

print("Ready. Data will be saved to:", DATA_DIR.resolve())

## 1. Aggregate CNC stats per city per year

For each city/year project we fetch:
- `total_observations` (total_results from observations endpoint)
- `unique_species` (total_results from observations/species_counts)
- `total_participants` (total_results from observations/observers)

In [ ]:
rows = []

for city, projects in CITIES.items():
    for year, project_slug in projects.items():
        print(f"Fetching {city} {year} ({project_slug})...")

        obs = api_get(f"{API_BASE}/observations?project_id={project_slug}&per_page=0")
        total_observations = obs["total_results"]

        species = api_get(f"{API_BASE}/observations/species_counts?project_id={project_slug}&per_page=0")
        unique_species = species["total_results"]

        observers = api_get(f"{API_BASE}/observations/observers?project_id={project_slug}&per_page=0")
        total_participants = observers["total_results"]

        species_per_observer = round(unique_species / max(total_participants, 1), 2)

        rows.append({
            "city": city,
            "year": year,
            "total_observations": total_observations,
            "unique_species": unique_species,
            "total_participants": total_participants,
            "species_per_observer": species_per_observer,
        })
        print(f"  -> {total_observations:,} obs, {unique_species:,} species, {total_participants:,} participants")

cnc_stats = pd.DataFrame(rows)
cnc_stats.to_csv(DATA_DIR / "cnc_city_stats.csv", index=False)
print(f"\nSaved {len(cnc_stats)} rows to cnc_city_stats.csv")
cnc_stats

## 2. Taxon breakdown per city (CNC 2025)

For each city we query observations filtered by `iconic_taxa` to get counts per major taxon group.

In [ ]:
TAXON_GROUPS = [
    "Plantae", "Insecta", "Aves", "Fungi", "Arachnida",
    "Reptilia", "Mollusca", "Mammalia", "Amphibia", "Actinopterygii",
]

taxon_rows = []

for taxon in TAXON_GROUPS:
    row = {"taxon_group": taxon}
    for city, projects in CITIES.items():
        slug = projects[2025]
        data = api_get(f"{API_BASE}/observations?project_id={slug}&iconic_taxa={taxon}&per_page=0")
        col_name = city.lower().replace(" ", "_")
        row[col_name] = data["total_results"]
        print(f"  {city} / {taxon}: {data['total_results']:,}")
    taxon_rows.append(row)

taxon_df = pd.DataFrame(taxon_rows)
taxon_df.to_csv(DATA_DIR / "cnc_city_taxon.csv", index=False)
print(f"\nSaved {len(taxon_df)} rows to cnc_city_taxon.csv")
taxon_df

## 3. Top 10 species per city (CNC 2025)

We use the `observations/species_counts` endpoint sorted by count to get the most-observed species.

In [ ]:
species_rows = []

for city, projects in CITIES.items():
    slug = projects[2025]
    print(f"Fetching top species for {city}...")
    data = api_get(f"{API_BASE}/observations/species_counts?project_id={slug}&per_page=10")

    for rank, result in enumerate(data["results"], start=1):
        taxon = result["taxon"]
        species_rows.append({
            "city": city,
            "rank": rank,
            "scientific_name": taxon.get("name", ""),
            "common_name": taxon.get("preferred_common_name", taxon.get("name", "")),
            "count": result["count"],
            "taxon_group": taxon.get("iconic_taxon_name", "Unknown"),
        })

    print(f"  -> Got {min(10, len(data['results']))} species")

species_df = pd.DataFrame(species_rows)
species_df.to_csv(DATA_DIR / "cnc_city_top_species.csv", index=False)
print(f"\nSaved {len(species_df)} rows to cnc_city_top_species.csv")
species_df

## Done

Three CSV files have been saved to `backend/data/`:

| File | Contents |
|---|---|
| `cnc_city_stats.csv` | Aggregate stats (observations, species, participants) per city per year |
| `cnc_city_taxon.csv` | Observation counts per taxon group per city (CNC 2025) |
| `cnc_city_top_species.csv` | Top 10 most observed species per city (CNC 2025) |

The backend router (`comparison.py`) loads these files at startup instead of using hardcoded data.